# Notebook 3 — Demand Classification (Syntetos–Boylan)

**Goals**

1. Understand why one-size-fits-all forecasting fails on diverse series.
2. Compute the two diagnostic numbers — **ADI** and **CV²** — for any
   series.
3. Place every forecast key in one of four quadrants: *smooth*,
   *intermittent*, *erratic*, *lumpy*.
4. Read the classification scatter plot and decide which model family
   fits which series.


In [1]:
# ──────────────────────────────────────────────────────────────────────────
# COLAB SETUP — run this once at the top of every tutorial notebook.
# It installs plotly + statsmodels and makes the toolkit importable.
# If you are running locally (not in Colab) the !pip line is harmless.
# ──────────────────────────────────────────────────────────────────────────
!pip install -q plotly statsmodels
import sys, os
# If you uploaded forecasting_toolkit.zip to Colab, unzip it once:
#   !unzip -o forecasting_toolkit.zip
# Otherwise place forecasting_toolkit/ next to this notebook.
sys.path.insert(0, os.path.abspath('.'))
import forecasting_toolkit as ft
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = 'colab'   # change to 'notebook' for local Jupyter


In [2]:
# ──────────────────────────────────────────────────────────────────────────
# POINT THIS AT YOUR DATASET — fill in the four lines below.
# Everything in this notebook works for ANY tabular sales/demand dataset
# (M5, Rossmann, Walmart Store Item Demand, custom CSVs, etc.).
#
#   DATA_PATH    – path to your CSV / Parquet file
#   DATE_COL     – name of the timestamp column
#   TARGET_COL   – name of the column you want to forecast
#   KEY_COLS     – list of columns that together identify ONE time series
#   STATIC_COLS  – columns constant within a key (e.g. store_type, category)
#   DYNAMIC_COLS – columns that vary in time within a key (e.g. promo, price)
#   FREQUENCY    – pandas offset alias: 'D','W','MS','H',…
# ──────────────────────────────────────────────────────────────────────────
DATA_PATH    = './datasets/rohlik_kaggle/sales_processed_tft.csv'
DATE_COL     = 'date'
TARGET_COL   = 'sales'
KEY_COLS     = ['unique_id']
STATIC_COLS  = ['warehouse','product_unique_id','name','L1_category_name_en','L2_category_name_en','L3_category_name_en','L4_category_name_en','country']
DYNAMIC_COLS = ['total_orders','sell_price_main','type_0_discount','type_1_discount','type_2_discount','type_3_discount','type_4_discount','type_5_discount','type_6_discount','holiday_name',
                'holiday','shops_closed','winter_school_holidays','school_holidays','weekday','week','month','day','is_month_start','is_month_end','quarter','weekend','days_since_2020']
FREQUENCY    = 'D'

from forecasting_toolkit import data_io
spec = data_io.make_spec(
    date_col=DATE_COL, target_col=TARGET_COL,
    key_cols=KEY_COLS, static_cols=STATIC_COLS,
    dynamic_cols=DYNAMIC_COLS, frequency=FREQUENCY,
)
df = data_io.load_data(DATA_PATH, spec)
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head()


Loaded 4,054,440 rows × 38 columns


,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,...,month,year,prev_year,day,is_month_start,is_month_end,quarter,weekend,days_since_2020,country
0,0,2022-07-18,Budapest_1,5289.0,3.97,710.89,0.09,0.0,0.0,0.00000,...,7,2022,2022,18,False,False,3,0,929,Hungary
1,0,2022-07-19,Budapest_1,5255.0,73.36,710.89,1.00,0.0,0.0,0.00000,...,7,2022,2022,19,False,False,3,0,930,Hungary
2,0,2022-07-20,Budapest_1,5334.0,558.09,710.89,0.96,0.0,0.0,0.45045,...,7,2022,2022,20,False,False,3,0,931,Hungary
3,0,2022-07-21,Budapest_1,5459.0,14.03,710.89,0.06,0.0,0.0,0.45045,...,7,2022,2022,21,False,False,3,0,932,Hungary
4,0,2022-07-22,Budapest_1,5461.0,558.53,710.89,0.97,0.0,0.0,0.45045,...,7,2022,2022,22,False,False,3,0,933,Hungary


## 3.1 Why classify?

Sales / demand series are **not** all alike. Compare these three:

- A best-selling SKU at a busy store: demand every day, modest
  variation. Easy.
- A spare part: zero demand most days, an order arrives once a fortnight.
  Average is low but *positive* sales are typical when they happen.
- A luxury good: orders are rare AND wildly different in size when they
  do happen. Some weeks ten units, some weeks one.

Forcing the same model on all three is a recipe for poor accuracy. The
**Syntetos–Boylan** scheme partitions every series with two numbers:

| Number | Measures |
|--------|----------|
| **ADI** (Average Demand Interval) | how *frequent* demand is |
| **CV²** of non-zero demand | how *variable* demand size is |


## 3.2 ADI — Average Demand Interval

$$\text{ADI} = \frac{\text{total periods}}{\text{periods with non-zero demand}}$$

- ADI ≈ 1 → demand happens every period (regular).
- ADI » 1 → demand happens sporadically.
- ADI = ∞ → no demand at all (the series is all zeros).

Implementation in the toolkit:

```python
ft.classification.compute_adi(series)
```


In [3]:
# Pick one example series to show the calculation step by step
keys = ft.data_io.summarize_keys(df, spec)
example_key = keys.iloc[0][KEY_COLS].to_dict()
single = ft.data_io.get_series(df, spec, example_key)[TARGET_COL]

adi = ft.classification.compute_adi(single)
total = len(single)
nonzero = (single != 0).sum()
print(f'Series: {example_key}')
print(f'  total periods       = {total}')
print(f'  non-zero periods    = {nonzero}')
print(f'  ADI = {total} / {nonzero} = {adi:.3f}')


Series: {'unique_id': 4755}
  total periods       = 1416
  non-zero periods    = 1416
  ADI = 1416 / 1416 = 1.000


## 3.3 CV² — Coefficient of Variation squared (of non-zero demand only)

$$\text{CV}^2 = \left(\frac{\sigma_\text{nz}}{\mu_\text{nz}}\right)^2$$

The non-zero filter is important: we already captured *occurrence*
variability with ADI, so CV² should isolate **size** variability.

- CV² small → when demand happens, the size is predictable.
- CV² large → demand size jumps around even when it does happen.


In [4]:
cv2 = ft.classification.compute_cv2(single)
nz = single[single != 0]
print(f'  non-zero mean = {nz.mean():.3f}')
print(f'  non-zero std  = {nz.std():.3f}')
print(f'  CV² = ({nz.std():.3f} / {nz.mean():.3f})² = {cv2:.3f}')


  non-zero mean = 117.569
  non-zero std  = 50.471
  CV² = (50.471 / 117.569)² = 0.184


## 3.4 The four-quadrant classification

The Syntetos–Boylan thresholds are **ADI = 1.32** and **CV² = 0.49**.
They split the (ADI, CV²) plane into four boxes:

|                    | **CV² < 0.49** (predictable size) | **CV² ≥ 0.49** (volatile size) |
|--------------------|-----------------------------------|---------------------------------|
| **ADI < 1.32** (frequent)  | **Smooth**       | **Erratic**       |
| **ADI ≥ 1.32** (sporadic)  | **Intermittent** | **Lumpy**         |

Forecasting strategy by class — at a glance:


In [5]:
for cls in ['smooth', 'erratic', 'intermittent', 'lumpy']:
    print(f'\n {cls.upper():>12s} — {ft.classification.recommend_strategy(cls)}')



       SMOOTH — Classical methods work well: ETS, ARIMA, regression with trend and seasonality. Standard error metrics (MAE/RMSE) are reliable.

      ERRATIC — High demand-size variance with regular occurrence: try robust regression, log/Box-Cox transforms before ETS/ARIMA, or quantile forecasting to protect against outliers.

 INTERMITTENT — Use Croston, SBA (Syntetos–Boylan Approximation), TSB or compound Poisson models. Avoid plain MAPE — many actuals are zero.

        LUMPY — The hardest category — irregular timing AND irregular size. Croston/SBA is the safe baseline; consider hierarchical aggregation upward to a smoother level for forecasting.


## 3.5 Classify ALL series in the dataset


In [6]:
classification = ft.classification.classify_all_series(df, spec)
print(f'Classified {len(classification):,} series.')
classification.head(10)


Classified 5,390 series.


,unique_id,n_obs,zero_share,adi,cv2,sb_class
0,0,35,0.000000,1.000000,0.706612,erratic
1,1,203,0.000000,1.000000,0.369278,smooth
2,2,65,0.000000,1.000000,0.081020,smooth
3,3,386,0.000000,1.000000,0.301501,smooth
4,5,483,0.010352,1.010460,0.099682,smooth
5,6,852,0.005869,1.005903,0.246108,smooth
6,7,1063,0.043274,1.045231,0.587517,erratic
7,8,714,0.001401,1.001403,0.676970,erratic
8,9,1394,0.001435,1.001437,0.259399,smooth
9,10,1395,0.000717,1.000717,0.225664,smooth


### Summary of the class distribution


In [7]:
summary = ft.classification.classification_summary(classification)
summary


,sb_class,n_series,pct
0,smooth,3719,69.00
1,erratic,1628,30.20
2,intermittent,28,0.52
3,lumpy,14,0.26
4,undefined,1,0.02


In [8]:
fig = ft.plotting.plot_sb_summary(summary)
fig.show()


## 3.6 The classification scatter plot

Each marker is one forecast key, placed at its (ADI, CV²) coordinates
and coloured by class. The two dashed grey lines mark the SB thresholds.


In [9]:
fig = ft.plotting.plot_sb_classification(classification)
fig.show()


The same plot with **log axes** is much more readable when you have
strong outliers (very high ADI or CV²):


In [10]:
fig = ft.plotting.plot_sb_classification(classification, log_axes=True,
        title='Syntetos–Boylan classification (log axes)')
fig.show()


## 3.7 Sample series from each quadrant

Eyeballing one example from each class makes the labels concrete.


In [11]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pick the most-populous example from each class
samples = {}
for cls in ['smooth', 'intermittent', 'erratic', 'lumpy']:
    in_cls = classification[classification['sb_class'] == cls]
    if len(in_cls) == 0:
        continue
    # Pick the median-length series in the class
    in_cls = in_cls.sort_values('n_obs')
    samples[cls] = in_cls.iloc[len(in_cls) // 2][KEY_COLS].to_dict()

# Plot a 2x2 grid
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=list(samples.keys()),
                    vertical_spacing=0.12, horizontal_spacing=0.08)
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
for (cls, key), (r, c) in zip(samples.items(), positions):
    sub = ft.data_io.get_series(df, spec, key).sort_values(DATE_COL)
    fig.add_trace(
        go.Scatter(x=sub[DATE_COL], y=sub[TARGET_COL], mode='lines',
                   line=dict(color=ft.plotting.SB_COLORS[cls], width=1.4),
                   name=cls, showlegend=False),
        row=r, col=c)
fig.update_layout(height=600, template='plotly_white',
                  title='One example series from each SB class')
fig.show()


## 3.8 Practical follow-up by class

For each class in your dataset, ask yourself:

| Class          | Action                                                         |
|----------------|----------------------------------------------------------------|
| **smooth**     | Standard ML / classical methods. Strong baselines.             |
| **intermittent** | Croston / SBA / TSB; use MASE not MAPE for accuracy.        |
| **erratic**    | Robust regression, log/Box-Cox, quantile forecasts.            |
| **lumpy**      | Hardest class — consider aggregating up the hierarchy.         |
| **no_demand**  | Forecast = 0; only revisit if external info changes.           |

You can save the classification table for downstream notebooks:


In [12]:
# Persist for reuse — uncomment if you want to write to disk:
# classification.to_csv('classification.csv', index=False)
print('Classification dataframe ready in variable `classification`')
classification.groupby('sb_class')['n_obs'].describe()


Classification dataframe ready in variable `classification`


,count,mean,std,min,25%,50%,75%,max
sb_class,,,,,,,,
erratic,1628.0,693.420147,472.177617,7.0,228.0,631.5,1144.25,1415.0
intermittent,28.0,817.500000,603.133085,8.0,173.0,880.5,1412.00,1412.0
lumpy,14.0,869.357143,488.215370,107.0,464.0,1096.0,1321.75,1412.0
smooth,3719.0,777.220489,510.265650,7.0,283.0,798.0,1352.00,1416.0
undefined,1.0,8.000000,NaN,8.0,8.0,8.0,8.00,8.0


## 3.9 Take-aways

- ADI and CV² are simple, well-defined, easy to compute — but they tell
  you almost everything about which forecasting model family to start
  with.
- The 4-quadrant scatter plot, with the dashed thresholds, is a great
  artefact to share with stakeholders early in any forecasting project.
- A dataset that is mostly **smooth** is *much* easier to forecast than
  one dominated by **lumpy** series. Knowing the mix sets accuracy
  expectations honestly.
